# PlacementIQ — Week 09
## SQL-First Streaming Simulation with Auto Loader

**ZENAIZ × BVRIT Hyderabad | Team 16 | PlacementIQ**

**Technology:** Databricks Free Edition · Spark SQL first · Light PySpark · Auto Loader · Delta

### Week 09 objective

Simulate incremental recruitment-data arrival for PlacementIQ using Auto Loader and Structured Streaming.

The notebook is deliberately **SQL-first**:
- SQL is used for file inspection, source counts, validation, Bronze checks, duplicates, summaries and reconciliation.
- Only the Auto Loader streaming source and `AvailableNow` streaming write use light PySpark.

### Recruitment flow

```text
PlacementIQ application JSON drops
        ↓
week09_landing/
        ↓
Auto Loader
        ↓
bronze_placementiq_application_events_stream
        ↓
SQL validation / reporting
        ↓
Power BI / downstream analytics
```

> Important: this notebook does not invent source counts. All counts shown after execution must come from your actual files.


## 1. What this notebook proves

By the end, you should be able to show:

1. The first application drop is processed.
2. A later drop is processed incrementally.
3. Re-running with no new file adds no rows.
4. Duplicate `application_id` values, if present in the source, remain visible for audit.
5. Source-file counts reconcile with the Bronze streaming table.
6. The Bronze table contains file lineage and ingestion time.
7. SQL can produce the recruitment metrics needed by the PlacementIQ Power BI dashboard.

This follows the **method** of the PageLoop example, but the entities are PlacementIQ recruitment applications.


## 2. PlacementIQ fields used

The supplied PlacementIQ data contains application fields such as:

- `application_id`
- `company_id`
- `role_id`
- `application_date`
- `application_status`
- `shortlist_flag`
- `final_outcome`
- `final_outcome_date`
- `rejection_reason`
- `last_updated_timestamp`

Interview fields should be added only if they are present in the actual source drop. Do not invent interview columns merely because the dashboard contains interview metrics.


In [0]:
%sql
USE CATALOG `placementiq-campus-career-analytics`;
USE SCHEMA `default`;

SELECT
  current_catalog() AS active_catalog,
  current_schema()  AS active_schema;


active_catalog,active_schema
placementiq-campus-career-analytics,default


In [0]:
%sql
CREATE VOLUME IF NOT EXISTS workspace.default.placementiq;

In [0]:
display(spark.sql("SHOW VOLUMES IN workspace.default"))

database,volume_name
default,placementiq


## 3. Configure the streaming folders

Create these folders in your Databricks Volume:

| Purpose | Path |
|---|---|
| Source drop files | `/Volumes/workspace/default/placementiq/week09_source/` |
| Landing folder | `/Volumes/workspace/default/placementiq/week09_landing/` |
| Checkpoint | `/Volumes/workspace/default/placementiq/week09_checkpoint/` |

Put only the files that should arrive during this streaming exercise in the **source** folder.

The landing folder should start empty.


In [0]:
%python
dbutils.fs.mkdirs("/Volumes/workspace/default/placementiq/week09_source/")
dbutils.fs.mkdirs("/Volumes/workspace/default/placementiq/week09_landing/")
dbutils.fs.mkdirs("/Volumes/workspace/default/placementiq/week09_checkpoint/")


True

In [0]:
%sql
LIST '/Volumes/workspace/default/placementiq/week09_source/';

LIST '/Volumes/workspace/default/placementiq/week09_landing/';


path,name,size,modification_time


In [0]:
%sql

LIST '/Volumes/workspace/default/placementiq/week09_source/';

path,name,size,modification_time


## 4. Inspect the first application drop before streaming

Replace the filename below with the **actual first JSON file** in your source folder.

This is an intentional SQL inspection step. It lets you confirm the real schema before defining the stream.


In [0]:
%sql
-- Inspect the first PlacementIQ interview status event JSON file.
SELECT *
FROM read_files(
  '/Volumes/workspace/default/placementiq/interview_status_event_drop_01_baseline.json',
  format => 'json'
)
LIMIT 10;


application_id,company_id,event_sequence,event_timestamp,event_type,ingestion_timestamp,interview_event_id,interview_id,interview_mode,interview_result,interview_status,role_id,round_number,schema_version,source_system,student_id,_rescued_data
A000099,C020,1,2026-06-13T09:00:00Z,scheduled,2026-06-13T10:05:00Z,E00000001,I0000302,HYBRID,null,SCHEDULED,R020-04,4,1.0,placementiq_stream_sim,S00099,null
A000099,C020,2,2026-06-15T09:00:00Z,started,2026-06-15T10:05:00Z,E00000002,I0000302,HYBRID,null,IN_PROGRESS,R020-04,4,1.0,placementiq_stream_sim,S00099,null
A000099,C020,3,2026-06-15T10:00:00Z,completed,2026-06-15T11:05:00Z,E00000003,I0000302,HYBRID,ADVANCED,COMPLETED,R020-04,4,1.0,placementiq_stream_sim,S00099,null
A000099,C020,4,2026-06-15T13:00:00Z,result_published,2026-06-15T14:05:00Z,E00000004,I0000302,HYBRID,REJECTED,RESULT_PUBLISHED,R020-04,4,1.0,placementiq_stream_sim,S00099,null
A006905,C094,1,2026-06-13T09:45:00Z,scheduled,2026-06-13T10:50:00Z,E00000005,I0021523,HYBRID,null,SCHEDULED,R094-05,4,1.0,placementiq_stream_sim,S01327,null
A006905,C094,2,2026-06-15T09:45:00Z,started,2026-06-15T10:50:00Z,E00000006,I0021523,HYBRID,null,IN_PROGRESS,R094-05,4,1.0,placementiq_stream_sim,S01327,null
A006905,C094,3,2026-06-15T10:45:00Z,completed,2026-06-15T11:50:00Z,E00000007,I0021523,HYBRID,ADVANCED,COMPLETED,R094-05,4,1.0,placementiq_stream_sim,S01327,null
A006905,C094,4,2026-06-15T13:45:00Z,result_published,2026-06-15T14:50:00Z,E00000008,I0021523,HYBRID,REJECTED,RESULT_PUBLISHED,R094-05,4,1.0,placementiq_stream_sim,S01327,null
A000303,C100,1,2026-06-13T10:30:00Z,scheduled,2026-06-13T11:35:00Z,E00000009,I0000929,ONSITE,null,SCHEDULED,R100-02,4,1.0,placementiq_stream_sim,S00303,null
A000303,C100,2,2026-06-15T10:30:00Z,started,2026-06-15T11:35:00Z,E00000010,I0000929,ONSITE,null,IN_PROGRESS,R100-02,4,1.0,placementiq_stream_sim,S00303,null


In [0]:
%sql
-- Source control total for Drop 01.
SELECT
  COUNT(*) AS physical_rows,
  COUNT(DISTINCT application_id) AS distinct_application_ids
FROM read_files(
  '/Volumes/workspace/default/placementiq/interview_status_event_drop_01_baseline.json',
  format => 'json'
);


physical_rows,distinct_application_ids
60,14


## 5. Define Auto Loader

Auto Loader is the streaming part of the notebook.

Because the exact source schema must be taken from the real PlacementIQ file, this version uses JSON schema inference for the raw Bronze stream. The SQL validation cells later check the business columns.

The stream watches **landing**, not source.


In [0]:
%python
from pyspark.sql.functions import col, current_timestamp, lit

applications_stream = (
    spark.readStream
         .format("cloudFiles")
         .option("cloudFiles.format", "json")
         .option("cloudFiles.inferColumnTypes", "true")
         .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
         .option("cloudFiles.schemaLocation", "/Volumes/workspace/default/placementiq/week09_schema/")
         .load("/Volumes/workspace/default/placementiq/week09_landing/")
         .select(
             "*",
             col("_metadata.file_path").alias("_source_file_path"),
             col("_metadata.file_name").alias("_source_file_name"),
             current_timestamp().alias("_ingested_at"),
             lit("APPLICATION").alias("_record_type")
         )
)

print("PlacementIQ Auto Loader source is ready.")


PlacementIQ Auto Loader source is ready.


## 6. Define the streaming write

`AvailableNow` means:

> Process all files that are available when the run starts, update the checkpoint, then stop.

This makes the notebook easy to demonstrate in a weekly internship exercise because you can show one drop arriving at a time.


In [0]:
%python
def run_available_now():
    query = (
        applications_stream.writeStream
            .option(
                "checkpointLocation",
                "/Volumes/workspace/default/placementiq/week09_checkpoint/"
            )
            .trigger(availableNow=True)
            .toTable(
                "`placementiq-campus-career-analytics`.default.bronze_placementiq_application_events_stream"
            )
    )
    query.awaitTermination()
    print("AvailableNow run finished.")


In [0]:
%python

display(dbutils.fs.ls("/Volumes/workspace/default/placementiq/week09_source/"))

path,name,size,modificationTime
dbfs:/Volumes/workspace/default/placementiq/week09_source/interview_status_event_drop_01_baseline.json,interview_status_event_drop_01_baseline.json,26910,1791349792000


In [0]:
%python

display(dbutils.fs.ls(
    "/Volumes/workspace/default/placementiq/week09_source/"
))

path,name,size,modificationTime
dbfs:/Volumes/workspace/default/placementiq/week09_source/interview_status_event_drop_01_baseline.json,interview_status_event_drop_01_baseline.json,26910,1791349792000


In [0]:
%python

display(dbutils.fs.ls(
    "/Volumes/workspace/default/placementiq/interview_status_event_drop_01_baseline.json"
))

path,name,size,modificationTime
dbfs:/Volumes/workspace/default/placementiq/interview_status_event_drop_01_baseline.json,interview_status_event_drop_01_baseline.json,26910,1791306421000


## 7. Drop 01 — first arrival

Copy only Drop 01 from the source folder into landing.

Do **not** copy Drop 02 yet.


In [0]:

%python

dbutils.fs.cp(
    "/Volumes/workspace/default/placementiq/week09_source/interview_status_event_drop_01_baseline.json",
    "/Volumes/workspace/default/placementiq/week09_landing/interview_status_event_drop_01_baseline.json"
)



True

In [0]:
%sql
LIST '/Volumes/workspace/default/placementiq/week09_landing/';


path,name,size,modification_time
/Volumes/workspace/default/placementiq/week09_landing/interview_status_event_drop_01_baseline.json,interview_status_event_drop_01_baseline.json,26910,1791350204000


In [0]:
%python

display(dbutils.fs.ls(
    "/Volumes/workspace/default/placementiq/week09_source/"
))


path,name,size,modificationTime
dbfs:/Volumes/workspace/default/placementiq/week09_source/interview_status_event_drop_01_baseline.json,interview_status_event_drop_01_baseline.json,26910,1791349792000


In [0]:
%python
run_available_now()


AvailableNow run finished.


In [0]:
%sql
SELECT
  _source_file_name,
  COUNT(*) AS physical_rows,
  COUNT(DISTINCT application_id) AS distinct_application_ids,
  MIN(event_timestamp) AS first_application_date,
  MAX(event_timestamp) AS last_application_date
FROM `placementiq-campus-career-analytics`.default.bronze_placementiq_application_events_stream
GROUP BY _source_file_name
ORDER BY _source_file_name;


_source_file_name,physical_rows,distinct_application_ids,first_application_date,last_application_date
interview_status_event_drop_01_baseline.json,60,14,2026-06-13T09:00:00Z,2026-06-15T23:30:00Z


## 8. Prove that Drop 01 was processed

The result above is your execution evidence.

Do not replace the returned counts with numbers from an example. Record the actual values in the Week 09 log.


In [0]:
%sql
SELECT
  COUNT(*) AS bronze_physical_rows,
  COUNT(DISTINCT application_id) AS bronze_distinct_application_ids,
  COUNT(DISTINCT _source_file_name) AS files_processed
FROM `placementiq-campus-career-analytics`.default.bronze_placementiq_application_events_stream;


bronze_physical_rows,bronze_distinct_application_ids,files_processed
60,14,1


## 9. Inspect Auto Loader file state

The checkpoint keeps track of files that have already been processed.

The exact availability of `cloud_files_state()` can depend on the Databricks runtime. If your runtime supports it, use the query below.


In [0]:
%sql
SELECT *
FROM cloud_files_state(
  '/Volumes/workspace/default/placementiq/week09_checkpoint/'
)
ORDER BY processed_time DESC;


path,size,create_time,discovery_time,commit_time,archive_time,source_id,processed_time,ingestion_state,archive_mode,move_location
dbstorage-prod-1wb5dtx4sf/uc/8dba43c0-c960-433e-a10e-c889b8f53110/570cbad5-dea8-4ddd-9dd0-706cf349395a/__unitystorage/catalogs/918f2cac-b848-4e59-9855-be68e527fd13/volumes/a8b5ab63-df63-460a-ac7d-5c139d0c388d/week09_landing/interview_status_event_drop_01_baseline.json,26910,2026-10-07T05:16:44.000Z,2026-10-07T05:19:48.994Z,null,null,0,2026-10-07T05:19:55.518Z,INGESTED,null,null


## 10. SQL validation of the recruitment fields

Use SQL to check the fields needed by the PlacementIQ dashboard.

This is intentionally a **business validation layer**, not a replacement for the Week 06 DQ rulebook.


In [0]:
%sql
SELECT
  COUNT(*) AS rows,
  COUNT_IF(application_id IS NULL OR TRIM(application_id) = '') AS missing_application_id,
  COUNT_IF(company_id IS NULL OR TRIM(company_id) = '') AS missing_company_id,
  COUNT_IF(role_id IS NULL OR TRIM(role_id) = '') AS missing_role_id,
  COUNT_IF(event_timestamp IS NULL) AS missing_application_date,
  COUNT_IF(interview_status IS NULL OR TRIM(interview_status) = '') AS missing_application_status
FROM `placementiq-campus-career-analytics`.default.bronze_placementiq_application_events_stream;


rows,missing_application_id,missing_company_id,missing_role_id,missing_application_date,missing_application_status
60,0,0,0,0,0


## 11. Application status summary

This is useful evidence for the Power BI recruitment dashboard.


In [0]:
%sql
SELECT
  interview_status,
  COUNT(*) AS applications
FROM `placementiq-campus-career-analytics`.default.bronze_placementiq_application_events_stream
GROUP BY interview_status
ORDER BY applications DESC;


interview_status,applications
COMPLETED,15
IN_PROGRESS,15
SCHEDULED,15
RESULT_PUBLISHED,15


## 12. Company-level recruitment summary

This matches the type of company/branch/application analysis used in your Power BI dashboard.

Use the actual company dimension when you have it; the Bronze query below is only an incremental-data check.


In [0]:
%sql
SELECT
  company_id,
  COUNT(DISTINCT application_id) AS total_applications,
  COUNT_IF(UPPER(interview_result) = 'ADVANCED') AS shortlisted_applications
FROM `placementiq-campus-career-analytics`.default.bronze_placementiq_application_events_stream
GROUP BY company_id
ORDER BY total_applications DESC;


company_id,total_applications,shortlisted_applications
C100,2,2
C083,2,2
C061,1,1
C057,1,1
C087,1,1
C043,1,1
C066,1,1
C020,1,1
C094,1,3
C006,1,1


## 13. Find duplicate application IDs

Do **not** automatically delete duplicates here.

A repeated application ID can represent multiple physical versions or repeated source data. Your Week 06 DQ rules and approved latest-update logic determine how it is handled downstream.


In [0]:
%sql
SELECT
  application_id,
  COUNT(*) AS physical_rows,
  COUNT(DISTINCT _source_file_name) AS source_files
FROM `placementiq-campus-career-analytics`.default.bronze_placementiq_application_events_stream
GROUP BY application_id
HAVING COUNT(*) > 1
ORDER BY physical_rows DESC, application_id;


application_id,physical_rows,source_files
A006905,8,1
A000099,4,1
A000303,4,1
A000545,4,1
A000934,4,1
A001239,4,1
A002821,4,1
A004623,4,1
A005717,4,1
A006002,4,1


## 14. Drop 02 — incremental arrival

Now copy only the second drop into landing and run `AvailableNow` again.

The checkpoint means Auto Loader should process the newly arrived file without reprocessing the previously processed file.


In [0]:
%python

dbutils.fs.cp(
    "/Volumes/workspace/default/placementiq/week09_source/interview_status_event_drop_02_progress.json",
    "/Volumes/workspace/default/placementiq/week09_landing/interview_status_event_drop_02_progress.json"
)


True

In [0]:
%sql
LIST '/Volumes/workspace/default/placementiq/week09_landing/';


path,name,size,modification_time
/Volumes/workspace/default/placementiq/week09_landing/interview_status_event_drop_01_baseline.json,interview_status_event_drop_01_baseline.json,26910,1791350204000
/Volumes/workspace/default/placementiq/week09_landing/interview_status_event_drop_02_progress.json,interview_status_event_drop_02_progress.json,35880,1791354305000


In [0]:
%python
run_available_now()


AvailableNow run finished.


In [0]:
%sql
SELECT
  _source_file_name,
  COUNT(*) AS physical_rows,
  COUNT(DISTINCT application_id) AS distinct_application_ids
FROM `placementiq-campus-career-analytics`.default.bronze_placementiq_application_events_stream
GROUP BY _source_file_name
ORDER BY _source_file_name;


_source_file_name,physical_rows,distinct_application_ids
interview_status_event_drop_01_baseline.json,60,14
interview_status_event_drop_02_progress.json,80,20


## 15. No-new-file test

Run the stream one more time **without adding a new file**.

The Bronze row count should remain unchanged.

This is one of the most important pieces of streaming evidence.


In [0]:
%sql
SELECT COUNT(*) AS before_no_new_file_run
FROM `placementiq-campus-career-analytics`.default.bronze_placementiq_application_events_stream;


before_no_new_file_run
140


In [0]:
%python
run_available_now()


AvailableNow run finished.


In [0]:
%sql
SELECT COUNT(*) AS after_no_new_file_run
FROM `placementiq-campus-career-analytics`.default.bronze_placementiq_application_events_stream;


after_no_new_file_run
140


## 16. Final source-to-Bronze reconciliation

For each source file, compare the physical rows read directly from the JSON files with the rows written to Bronze.

Do not reconcile using distinct application IDs because one application may legitimately have multiple physical versions.


In [0]:
%sql
SELECT
  _source_file_name,
  COUNT(*) AS bronze_physical_rows,
  COUNT(DISTINCT application_id) AS bronze_distinct_application_ids
FROM `placementiq-campus-career-analytics`.default.bronze_placementiq_application_events_stream
GROUP BY _source_file_name
ORDER BY _source_file_name;


_source_file_name,bronze_physical_rows,bronze_distinct_application_ids
interview_status_event_drop_01_baseline.json,60,14
interview_status_event_drop_02_progress.json,80,20


## 17. Latest application version for analytics

For a clean reporting view, use the approved `last_updated_timestamp` logic.

This is a reporting view only. Do not use it to delete or overwrite Bronze history.


In [0]:
%sql
CREATE OR REPLACE VIEW vw_placementiq_latest_applications AS
WITH ranked AS (
  SELECT
    *,
    ROW_NUMBER() OVER (
      PARTITION BY application_id
      ORDER BY
        CAST(ingestion_timestamp AS TIMESTAMP) DESC,
        CAST(event_timestamp AS TIMESTAMP) DESC,
        _source_file_name DESC
    ) AS rn
  FROM `placementiq-campus-career-analytics`.default.bronze_placementiq_application_events_stream
  WHERE application_id IS NOT NULL
)
SELECT *
FROM ranked
WHERE rn = 1;


In [0]:
%sql
SELECT
  COUNT(*) AS latest_application_records,
  COUNT(DISTINCT application_id) AS distinct_applications
FROM vw_placementiq_latest_applications;


latest_application_records,distinct_applications
31,31


## 18. Power BI-ready summary

This query gives a simple dataset for recruitment dashboard validation.


In [0]:
%sql
SELECT
  CAST(event_timestamp AS DATE) AS application_date,
  company_id,
  interview_status,
  COUNT(DISTINCT application_id) AS total_applications,
  COUNT(DISTINCT CASE
      WHEN UPPER(interview_result) = 'ADVANCED' THEN application_id
  END) AS shortlisted_applications
FROM vw_placementiq_latest_applications
GROUP BY
  CAST(event_timestamp AS DATE),
  company_id,
  interview_status
ORDER BY application_date, company_id;


application_date,company_id,interview_status,total_applications,shortlisted_applications
2026-06-15,C006,RESULT_PUBLISHED,1,0
2026-06-15,C014,RESULT_PUBLISHED,1,0
2026-06-15,C020,RESULT_PUBLISHED,1,0
2026-06-15,C043,RESULT_PUBLISHED,1,0
2026-06-15,C057,RESULT_PUBLISHED,1,0
2026-06-15,C061,RESULT_PUBLISHED,1,0
2026-06-15,C066,RESULT_PUBLISHED,1,0
2026-06-15,C083,RESULT_PUBLISHED,2,0
2026-06-15,C084,RESULT_PUBLISHED,1,0
2026-06-15,C094,RESULT_PUBLISHED,1,1


## 19. Interview data — only if your source contains it

Your Power BI page uses interview metrics such as completed/selected interviews.

Do **not** add invented interview columns to the streaming schema.

If interview JSON drops are supplied separately, create a second Auto Loader stream using the same pattern:

```text
interview JSON drops
        ↓
interview landing folder
        ↓
Auto Loader
        ↓
bronze_placementiq_interview_events_stream
```

Then validate `interview_id`, `application_id`, `round_number`, interview status/result and timestamps against the approved Week 06 DQ contract.


## 20. Week 09 evidence checklist

- [ ] Source folder contains the real PlacementIQ drop files.
- [ ] Landing starts empty.
- [ ] Drop 01 is processed.
- [ ] Drop 02 is processed only after it arrives.
- [ ] No-new-file run adds zero rows.
- [ ] Bronze contains source file lineage.
- [ ] Duplicate application IDs are visible rather than silently deleted.
- [ ] Application status summary is produced.
- [ ] Company-level application summary is produced.
- [ ] Latest-version reporting view is created using the approved update timestamp.
- [ ] Source-to-Bronze physical-row reconciliation is zero variance.
- [ ] Actual execution counts are recorded in the Week 09 log.
- [ ] Notebook is committed to GitHub.


## 21. Important implementation note

This notebook is **SQL-first**, not SQL-only.

Databricks Auto Loader's interactive streaming source and `writeStream` are most clearly expressed with a small PySpark block. Everything that can reasonably be SQL is SQL.

That is the same design principle as the PageLoop example you provided, adapted to PlacementIQ recruitment data.


## 22. AI Transparency Note

AI was used to adapt the PageLoop streaming-learning structure to the PlacementIQ recruitment project and to suggest SQL/PySpark implementation patterns.

The team must verify:
- actual source filenames and schemas;
- actual Databricks catalog/schema;
- actual application and interview columns;
- source counts;
- execution results;
- DQ and latest-version rules.

No example count should be presented as an actual PlacementIQ result until the notebook has been executed.
